# **Assignment 3: Introduction to Deep Learning for Sound Event Classification**
### Due: Thursday, March 12th, midnight EST
### Quiz date: Tuesday, March 17th in-class

CS-GY 6933: Machine Listening Spring 2026

Below you will find a mix of coding questions and writing questions to familiarize you with the fundamentals of signal processing in Python.

The assignment will have two parts:

1. Part 0: **Tutorial**: getting used to working with PyTorch (tensors, data loading, training and evaluating a simple model). This portion is not worth any points and typically will not have anything to "fill-in". I recommend you walk through the code and run it to understand the pieces before moving on if you are new to this material.

2. Parts 1-3: **Problem solving**: MFCC feature design (2pts), and model design for sound event classification (3pts). Totaling 5 points. The in-class quiz about the assignment will also be worth 5 points. There is also 1 point of extra credit available in this section!⭐


**A note on GPU Colab credits:** we can get a free student Pro membership to Colab that has 100 compute credits per month. Using an L4 GPU will get you plenty fast performance for these experiments and only uses 1.71 credits per hour (though don't turn to GPU runtime until the model training to save credits).


When you complete the assignment, please evaluate the notebook so that all results are shown/printed before submitting via Brightspace.



🚨 Please refrain from using ChatGPT etc. to fully write the code for this assignment. You will need to understand the content to succeed in the in-class quiz.

# Part 0: PyTorch fundamentals [0pts]
We'll be using PyTorch in this course for deep learning! In this section we will walk through some basics of Torch, setting up a dataset/"dataloader", and a general template for training a model. For more tutorials on PyTorch basics, check out the Torch website [here](https://pytorch.org/tutorials/beginner/basics/intro.html).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from tqdm import tqdm

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

### PyTorch Tensors
Tensors are the core data type you will work with in PyTorch. PyTorch tensors and NumPy arrays are both multi-dimensional data structures used for numerical computations, but tensors are optimized for GPU acceleration and support automatic differentiation (autograd), making them ideal for deep learning. NumPy arrays are primarily CPU-based and lack built-in support for gradients. Check out some examples of tensor manipulations below:

In [ ]:
# Creating a Tensor (like a NumPy array but with GPU support)
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # 2x2 matrix
print(x)

# Creating a Random Tensor
rand_tensor = torch.rand(3, 3)  # 3x3 matrix with random values
print(rand_tensor)

# Basic Tensor Operations
y = x + 2  # Element-wise addition
z = x * y  # Element-wise multiplication
print("Added Tensor:\n", y)
print("Multiplied Tensor:\n", z)

# Moving Tensors to GPU (if available)
device = "cuda" if torch.cuda.is_available() else "cpu" # You may see this line of code at the top of all
x_gpu = x.to(device)  # Moves tensor to GPU
print("Tensor on device:", x_gpu.device)

# Reshaping a Tensor
reshaped = x.view(4, 1)  # Reshape to 4x1
print("Reshaped Tensor:\n", reshaped)

# Converting Between NumPy and Torch
numpy_array = x.numpy()  # Convert to NumPy
torch_tensor = torch.from_numpy(numpy_array)  # Convert back to PyTorch tensor

tensor([[1., 2.],
        [3., 4.]])
tensor([[0.6581, 0.3443, 0.7620],
        [0.6399, 0.8609, 0.5154],
        [0.9596, 0.6096, 0.6305]])
Added Tensor:
 tensor([[3., 4.],
        [5., 6.]])
Multiplied Tensor:
 tensor([[ 3.,  8.],
        [15., 24.]])
Tensor on device: cpu
Reshaped Tensor:
 tensor([[1.],
        [2.],
        [3.],
        [4.]])


### Datasets and dataloaders
In PyTorch, we often work with iterable datasets, which are wrapped in a dataloader class. This provides a nice way to iterate through our data as we train our model. See the docs for more info here.

Below I've provided a very simple template dataset and dataloader to give you a framework to build off of as we get into the audio datasets below.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Our simple dataset class
class SimpleDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data  # (ex: numpy array of (n_samples, n_features), list of filepaths etc.)
        self.labels = labels  # (ex: list of class labels)

        print(f'Number of data samples: {len(self.data)}')

    def __len__(self):
        return len(self.data)  # Return dataset size, this is just a formality

    # Get item is the core method that retrieves one sample (and a label, optionally)
    def __getitem__(self, idx):
        # idx is the index into your full dataset (e.g. sample at index 2 from your dataset)
        return self.data[idx], self.labels[idx]  # Return sample and label


In [ ]:
# Dummy data
data = torch.randn(100, 10)  # 100 samples, each with 10 features
labels = torch.randint(0, 2, (100,))  # 100 binary labels

# Create dataset and DataLoader
dataset = SimpleDataset(data, labels)

# Dataloader is an iterable wrapper class for dataset
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Iterate over DataLoader
for batch in dataloader:
    inputs, targets = batch
    print(inputs.shape, targets.shape)
    break  # Stop after first batch for demo

# Note that if you are using different train/val/test splits, usually
# you create separate dataset instances+dataloaders for each of the splits

Number of data samples: 100
torch.Size([4, 10]) torch.Size([4])


### Model definition
Below we define a simple single-layer model class in PyTorch. Read through the comments to understandn the __init__ and forward() functionalities. Note that the weights and biases of your model layers are "under the hood" below - we won't be explicitly defining them here.

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, input_channels, n_classes):
        super(SimpleModel, self).__init__() # formality
        # Here we define the model architecture
        # This *doesn't* tell us how data flows through the model, just the architecture

        # Define one single linear layer
        # This has to accept 2-dim input and return n_classes output
        # Input should be (batch_size, input_channels)
        # Linear layer operates on channel dimension only
        self.fc = nn.Linear(input_channels, n_classes)

    # The forward pass is the core method in a model class
    # This determines how data x flows through the network
    def forward(self, x):
        output = self.fc(x)
        return output

In [ ]:
# Simple function to get the model config and number of parameters
def print_model(model):
    # Print model's state_dict
    print("Model's state dictionary (stored weights):")
    for param_tensor in model.state_dict():
        print("  ", param_tensor, "\t", tuple(model.state_dict()[param_tensor].size()))

    # Print the number of parameters in the model
    parameter_count =  sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("In total, this network has ", parameter_count, " trainable parameters")


In [ ]:
print_model(SimpleModel(input_channels=2, n_classes=10))

Model's state dictionary (stored weights):
   fc.weight 	 (10, 2)
   fc.bias 	 (10,)
In total, this network has  30  trainable parameters


In [ ]:
# Let's test out passing some dummy data through the model
# This is only a *forward* pass through the model e.g. "inference" - not training or doing any back propogation
model = SimpleModel(input_channels=2, n_classes=5)
sample_data = torch.randn(10,2)
output = model(sample_data)
print(output.shape)

torch.Size([10, 5])


### Backpropogation and optimization in pytorch
In neural networks, the model will learn to map inputs through intermediate ("hidden") representations of varying dimensions. This complexity helps our model learn more complex mapping functions, but adds a level of difficulty in figuring out how to update the weights of all of our parameters of the model (e.g. with gradient descent!).

To update our network given a target and predicted output, we will compute the loss (e.g. mean-squared error) using a differentiable loss function, and then compute the gradient of the loss function with respect to each model weight, and performing a small update in the opposite direction of the gradient. The computation of these gradients is called backpropagation, and allows us to systematically train large and complex neural networks.

Luckily, PyTorch provides automatic differentiation (e.g. autograd), which provides a built-in gradient computation for us! Let's play with a few aspects of gradients in PyTorch before incorporating the model component.

In [ ]:
# Note that below, we aren't actually doing any "model training"
# This shows how we would do manual optimization - without using built-in optimization yet.

# Sample tensor
x = torch.ones(5)
print(f"Creating a tensor of type {type(x)} with shape {x.shape}")
print(f"Starting x: {x}")

# During backpropagation, gradients will only be computed for tensors with the
# `requires_grad` attribute set to True. We can set this manually if need be
print(f"Does our tensor require gradient computation? {x.requires_grad}")
x.requires_grad = True
print(f"Does our tensor require gradient computation? {x.requires_grad}")

# To perform backpropagation, we need to complete a "forward pass" in which
# computations are performed on Tensor objects to compute a scalar loss value
# This is just a dummy scalar loss function - in practice this will
loss = 10 - x.sum()
print(f"Starting `loss` value: {loss}")
print(f"Gradients of x: {x.grad}") # no gradients yet
print(f"Loss function requires_grad?: {loss.requires_grad}")

# PyTorch will compute all required gradients for tensors involved in the
# computation of a scalar loss value once we call `.backward()`
loss.backward()
print(f"Gradients of x: {x.grad}")

# We can manually update our `weights` in the opposite direction of this gradient
# to reduce our loss value!
x = x - x.grad
print(f"Updated x: {x.data}")
loss = 10 - x.sum()
print(f"Updated `loss` value: {loss}")

Creating a tensor of type <class 'torch.Tensor'> with shape torch.Size([5])
Starting x: tensor([1., 1., 1., 1., 1.])
Does our tensor require gradient computation? False
Does our tensor require gradient computation? True
Starting `loss` value: 5.0
Gradients of x: None
Loss function requires_grad?: True
Gradients of x: tensor([-1., -1., -1., -1., -1.])
Updated x: tensor([2., 2., 2., 2., 2.])
Updated `loss` value: 0.0


n the above example we computed differentiable a scalar loss, used backpropagation to compute the gradients of the loss with respect to our "weights," and performed a gradient-based update on our weights to reduce the loss. Rather than managing the weight-update process by hand, we can defer to a built-in optimizer object that automatically adjusts weights based on stored gradients and standard hyperparameters (e.g. learning rate). When training neural networks with large numbers of parameters, this becomes much simpler than manually updating each weight. ([Credit](https://github.com/interactiveaudiolab/course-deep-learning/blob/main/notebooks/notebook_2_nn.ipynb))

In [ ]:
# Repeat our simple optimization, this time using the optimizer.
x = torch.ones(5).requires_grad_(True)
print(f"Starting x: {x}")

# Create an optimizer object and pass it an Iterable containing our "weights".
# Here, SGD is the torch stochastic gradient descent optimizer.
# It has been handed our tensor x as something to optimize and the learning rate
# (lr) is set to 1, which determines the step size for making changes to x.
# Note that this example learning rate is very high! in practice we usuallyuse something like 0.1 or 0.01.
opt = torch.optim.SGD([x], lr = 1.0)

# Compute loss and perform backpropagation.
loss = 10 - x.sum()
loss.backward()

# perform an automatic optimization step, i.e. a gradient-based update of our weights
opt.step()

print(f"Updated x: {x}")

Starting x: tensor([1., 1., 1., 1., 1.], requires_grad=True)
Updated x: tensor([2., 2., 2., 2., 2.], requires_grad=True)


### Simple training pipeline
Finally, let's put all of the pieces together and write a simple training script for a dummy classification task.

In [ ]:
# Data setup
train_data = torch.randn(100, 10, dtype=torch.float32)  # 100 samples, each with 10 features
train_labels = torch.randint(0, 5, (100,))  # 100 labels between 0-4

# Create dataset and DataLoader
train_dataset = SimpleDataset(train_data, train_labels)

# Dataloader is an iterable wrapper class for dataset
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Also create the validation dataloader
val_data = torch.randn(20, 10, dtype=torch.float32)
val_labels = torch.randint(0, 5, (20,))
val_dataset = SimpleDataset(val_data, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Model setup
model = SimpleModel(input_channels=10, n_classes=5)

# Optimizer setup, on all of our model parameters
opt = torch.optim.SGD(model.parameters(), lr = 0.01)

# Define the loss function
loss_fn = nn.CrossEntropyLoss().to(device)


# This tells your model that you are in training mode and not testing mode
# For our simple case this doesn't do much, but more complex layers such as Dropout
# behave differently in training vs. evaluation mode
model.train()


# TRAINING LOOP

# Loop through the entire [training] data each "epoch"
# So you will go through every "batch" of data inside this epoch
epochs = 10
for epoch in range(epochs):
    total_loss = 0.0
    for batch in train_dataloader:
        inputs, targets = batch
        # Set the gradients to 0 before running the network on the data each iteration, so that
        # loss gradients can be computed correctly during backpropagation
        opt.zero_grad()
        # print(f"Model input shape (batch): {inputs.shape}")

        # Get the output of the network on the data
        # Note that these are probabilities, *not* class labels!
        output = model(inputs)
        # print(f"Model output shape (batch): {output.shape}")

        # Measure the "loss" using mean squared error
        loss = loss_fn(output, targets)

        # This calculates the gradients, performing backpropagation to propagate
        # errors backward through the network's weights
        loss.backward()

        # This updates the network weights based on the freshly-computed gradient
        # now stored alongside each weight
        opt.step()

        # Accumulate the total loss per epoch
        total_loss += loss.item()

    print(f"Training Epoch {epoch+1}/{epochs}, Train Loss: {total_loss / len(dataloader):.4f}")

    # VALIDATION LOOP
    # The validation loop
    model.eval()
    total_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():  # Disable gradient computation
        for batch in val_dataloader:
            inputs, targets = batch
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            total_val_loss += loss.item()

            # Compute validation accuracy
            _, predicted = torch.max(outputs, dim=1)
            correct_val += (predicted == targets).sum().item()
            total_val += targets.size(0)

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_accuracy = correct_val / total_val

    # Print results
    print(f"Validation Epoch {epoch+1}/{epochs}, Val: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}\n")


Number of data samples: 100
Number of data samples: 20
Training Epoch 1/10, Train Loss: 1.7603
Validation Epoch 1/10, Val: 1.6248, Val Accuracy: 0.2000

Training Epoch 2/10, Train Loss: 1.7252
Validation Epoch 2/10, Val: 1.6170, Val Accuracy: 0.2000

Training Epoch 3/10, Train Loss: 1.6944
Validation Epoch 3/10, Val: 1.6102, Val Accuracy: 0.2500

Training Epoch 4/10, Train Loss: 1.6667
Validation Epoch 4/10, Val: 1.6048, Val Accuracy: 0.2500

Training Epoch 5/10, Train Loss: 1.6417
Validation Epoch 5/10, Val: 1.6001, Val Accuracy: 0.2500

Training Epoch 6/10, Train Loss: 1.6198
Validation Epoch 6/10, Val: 1.5964, Val Accuracy: 0.2500

Training Epoch 7/10, Train Loss: 1.5986
Validation Epoch 7/10, Val: 1.5937, Val Accuracy: 0.2500

Training Epoch 8/10, Train Loss: 1.5812
Validation Epoch 8/10, Val: 1.5914, Val Accuracy: 0.3000

Training Epoch 9/10, Train Loss: 1.5639
Validation Epoch 9/10, Val: 1.5901, Val Accuracy: 0.3000

Training Epoch 10/10, Train Loss: 1.5490
Validation Epoch 10/10

### Model evaluation
Now that we've "trained" our dummy model, let's walk set up the evaluation script. Note that typically you will have a training and validation loop, which run in sequence for N epochs, and once you have the final trained model, then you will do the final evaluation on the test set separately - similar to what we did with cross validation in assignment 2. But for this demo, we'll just do the training above and then use that trained model for a fake test loop.

In [ ]:
# Create dataset and DataLoader
test_data = torch.randn(20, 10, dtype=torch.float32)
test_labels = torch.randint(0, 5, (20,))
test_dataset = SimpleDataset(test_data, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True)

# Switch model to eval mode (IMPORTANT!)
model.eval()  # Set the model to evaluation mode

num_correct = 0

# Explicitly stop gradient computation here
with torch.no_grad():
    for batch in test_dataloader:
        inputs, targets = batch
        output = model(inputs)

        # Here we don't care as much about the loss (though you can still compute it)
        # Convert the output logits to class probabilities
        # Why didn't we do this argmax in training? CrossEntropy loss under the hood uses softmax :)
        predictions = torch.argmax(output, dim=-1)
        num_correct += (predictions == targets).sum().item()  # Count correct predictions

print(f"Test Accuracy: {num_correct / len(test_dataset)}")


Number of data samples: 20
Test Accuracy: 0.15


⚔️⚔️⚔️ Awesome! You are now equipped with the basic tools needed to start training real deep learning models! ⚔️⚔️⚔️

# Part 1: MFCC Feature Design [2pts]
In this section you'll be building an MFCC feature extractor as we discussed in class. You'll use this feature later in model training. Before we get into the details, first let's explore the dataset we'll be using in this assignment.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display
from scipy.fft import dct, idct

import librosa
import librosa.display
from IPython.display import Audio, display

### Data
We'll be using **ESC-50**, a popular dataset for environmental sound classification, for this assignment. The dataset has 2000 x 5-second environmental sound recordings across 50 classes. You can read more about the dataset [here](https://github.com/karolpiczak/ESC-50).

In [ ]:
# TODO: Run this code to download and unzip the dataset. It is about 6
# Download ESC-50 master zip. The zip file is about 600MB (should take ~2 min to run this cell)
!wget --progress=bar:force https://github.com/karoldvl/ESC-50/archive/master.zip

# Unzip
!unzip -q master.zip

# Rename folder (optional, cleaner path)
!mv ESC-50-master ESC50

# Remove zip file (optional)
!rm master.zip

print("Done!")

--2026-03-03 16:42:56--  https://github.com/karoldvl/ESC-50/archive/master.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/karolpiczak/ESC-50/archive/master.zip [following]
--2026-03-03 16:42:57--  https://github.com/karolpiczak/ESC-50/archive/master.zip
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/karolpiczak/ESC-50/zip/refs/heads/master [following]
--2026-03-03 16:42:57--  https://codeload.github.com/karolpiczak/ESC-50/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 140.82.114.10
Connecting to codeload.github.com (codeload.github.com)|140.82.114.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘master.zip’

master.zip              [   

#### 🔎 But first: check out the data 🔎
It's always a good first step to explore a new dataset and make sure the input data is in a format you're expecting.

**Deliverables:**
1. Load an audio file from ESC-50 (in the audio directory) with the default sample rate.
2. Plot the waveform.
3. Plot the log mel spectrogram (use `librosa`).
4. Play the audio.
5. Load the full metadata csv (in `esc50.csv`), locate the sample you're examining, and confirm that your label matches what you're expecting, and what you see in the spectrogram.


In [ ]:
# 🚨TODO🚨

#### Quick check: class distribution
Before we move on, let's also look at some aggregate statistics about the classes in the dataset:

**Deliverables**:
1. Print the number of files total.
2. Print the number of classes total.
3. Plot the number of files per class in a bar chart. For your own understanding: are the classes balanced? Unbalanced?

In [ ]:
# 🚨TODO🚨

### Designing MFCC features [1pt]
**Deliverables**

- Implement an MFCC feature extraction method as discussed in class.
    - You may ***not*** use librosa's built in MFCC method for this.
    - You ***may*** use numpy or scipy built-in methods as helper functions, and you can use the MelSpectrogram method from librosa.
- Using an ESC-50 audio file, plot the log-mel Spectrogram and MFCCs of this signal using your method. The MFCC plot should contain time on the x-axis and coefficient index on y-axis.


In [ ]:
def get_mfcc_features(mel_spec, m=None):
    """ Compute MFCC features from a mel spectrogram.
    Always drop the first 2 coefficients, and use `m`
    to decide how many remaining coefficients to keep after those are dropped.
    i.e., keep up to coefficient `m` (index m+2 before dropping [0-1]

    Tips:
    - Use DCT-Type II
    - Include norm="ortho" in your DCT method call for normalization.

    Parameters
    ----------
    mel_spec : np.ndarray, shape (n_mels, n_frames)
        Mel spectrogram (in power).
    m: Discard cepstral coefficients beyond this index m.

    Returns
    -------
    mfcc : np.ndarray, shape (n_mfcc, n_frames)
        Matrix of retained MFCC features.

    Returns:
      - mfcc_trunc: (m, n_frames) MFCC features, with appropriate coefficients dropped.
      - mfcc_trunc_full:(n_mels, n_frames) same information as mfcc_trunc, but of the original length.
        (dropped coefficients zeroed, that we'll use for envelope plots)
      - mel_db: (n_mels, n_frames) log-mel spectrogram
    """
    mfcc_trunc = None
    mfcc_trunc_full = None
    mel_db = None
    return mfcc_trunc, mfcc_trunc_full, mel_db


In [ ]:
# 🚨TODO🚨
# Code to test and plot your MFCC method
# Use these default parameters:
n_fft = 4096
window_length = 2048
hop_length = 1024
n_mels = 128
sr = 22050
fmin = 200
fmax = 8000

### Understanding MFCC: plotting the spectral envelope [1pt]
**Deliverables:**

To get a better understanding of the effect of MFCC transformation, create the following sequence of plots, for the audio file **ESC50/audio/5-198411-A-20.wav**:
1. Log Mel spectrogram
2. MFCC
3. Spectral envelope: log mel *spectrum* and reconstructed spectrum from your MFCCs, both at frame 54, overlayed on the same plot. Show 4 versions of this plot, for `m={5,10,12,40}` in your MFCC function to observe the effect.

**Guidelines and tips:**
- Take care in naming your axis appropriately and ensuring that the scaling of each plot makes sense - we will be looking for this as one aspect of grading here.
- Tip 1: You will want to reconstruct using the full-length MFCCs (containing zero'd coefficients).
- Tip 2: Because the first two coefficients are dropped, your enevelope may appear DC-shifted when reconstructed. To get around this you can add the mean of the log mel spectrogram back to the reconstructed spectrum

In [ ]:
# Use these default parameters
n_fft = 4096
window_length = 2048
hop_length = 1024
n_mels = 128
sr = 22050
fmin = 200
fmax = 8000

# 🚨TODO 🚨 your code and plots for the above.


# Part 2: Sound Event Classification [3 pts + 1 pt extra credit available]
You will design, train, and evaluate three model configurations for sound event classification:

The models:
1. MLP, using MFCC features
2. MLP, using log Mel spectrogram features
3. CNN (**1D**), using log Mel spectrogram features

Extra credit [1pt]:
1. CNN (**2D**), using log Mel spectrogram features. If you are doing the extra credit, follow the deliverables for the prior 3 models here, as well as the written question for full credit.

Some tips on the components of the pipeline you'll need to setup:
- Dataloader for ESC-50, retrieves (MFCC/Mel feature, label) per file. Handles train/validation/test splits by fold specified in ESC metadata.
- Model architecture for each.
- Training and validation pipeline.
- Evaluation given a trained model on the test set.




#### **Deliverables**


**1. Model investigation across features and architectures** [2.5 pts]
- Train the three model configurations above for multi-class sound event classification with ESC-50.
- Report **overall accuracy** of each model and **F1-score** of the top and bottom-3 performing classes per configuration (over 3-fold cross validation).

**2. Explanation of results and parameter choices for each model configuration** [0.5 pts]
- Brief written explanation justifying (1) model design/parameter choices for each of the three configurations, and (2) intuition about model performance. See prompt below for details.

**Guidelines for model design:**
- Final model configurations should be reported using **3-fold cross-validation**, in which 3 train/validation/test splits are chosen at random. ESC-50's metadata file has each audio clip labeled with a fold number already (Fold #1-5). Ideally we would do 5-fold, but let's stick with 3 for the scope of this assignment for computational purposes. Feel free to run individual folds (or smaller scale overfitting-style experiments) for early experimentation.
- This is an exercise in ***careful architecture choices for a simple model*** for the task and features at hand. The focus should be on an intentional and informed choices of layer parameters (e.g. hidden dimensions, pooling, model initialization, etc.)
- Model selection should be in terms of minimum validation loss.
- Your model parameter counts should not exceed 200k.
- You will likely want to use some feature normalization as discussed in class.


Baselines:
- As a reference, I was able to achieve ~20-305% test accuracy (depending on test fold) using a 90k parameter simple MLP+MFCC model. I used a learning rate of 1e-3, batch size 16, and trained for 10 epochs.


## 1: Model investigation and reports - 💡YOUR CODE HERE💡
Model configurations:
1. MLP, using MFCC features
2. MLP, using log Mel spectrogram features
3. CNN (**1D**), using log Mel spectrogram features

Cleary report results (refer above for what is required).

In [ ]:
# 🚨TODO🚨 - Freeform code for this section.
# Up to you how you want to structure things!

In [ ]:
# ----Use these defaults again-----
# Dataset
# Defaults:
# n_fft = 4096
# window_length = 2048
# hop_length = 1024
# n_mels = 128
# sr = 22050
# fmin = 200
# fmax = 8000
# -------------------------

In [ ]:
# OPTIONAL EXTRA CREDIT: 🚨TODO🚨

## 2: Design choice and results written response 💡YOUR RESPONSE HERE💡
🚨TODO🚨

***In your own words:***
1. For each of the three model configurations above, briefly explain and motivate your architectural design choices.
2. Is there one model configuration that outperforms the rest significantly? Why do you think so (or if not, why not)?


**</YOUR RESPONSE HERE/>**
